<a href="https://colab.research.google.com/github/gyan1131/Quantum_Computing_2026/blob/main/CPU%20affinity%20issues%20%232414.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 3.9 MB/s eta 0:00:00


In [2]:
from __future__ import annotations

import os

print(os.sched_getaffinity(0))
from qiskit_aer import AerSimulator

print(os.sched_getaffinity(0))

{0, 1}
{0, 1}


In [3]:
import os

print("Initial CPU affinity for the current process:", os.sched_getaffinity(0))

# Set the environment variables as described
os.environ["OMP_PROC_BIND"] = "close"
os.environ["OMP_PLACES"] = "cores"

print(f"\nEnvironment variable OMP_PROC_BIND set to: {os.environ.get('OMP_PROC_BIND')}")
print(f"Environment variable OMP_PLACES set to: {os.environ.get('OMP_PLACES')}")

# Check CPU affinity again after setting environment variables
# Note: Changing OMP_* environment variables within a running process
# might not immediately affect the process's current CPU affinity as reported by os.sched_getaffinity,
# because these are primarily hints for OpenMP runtime to bind threads within parallel regions.
# Their effect on the overall process affinity is often realized when set *before* the process starts,
# or for new child processes/threads that are created after the variables are set.
print("\nCPU affinity for the current process after setting OMP_PROC_BIND and OMP_PLACES:", os.sched_getaffinity(0))

# To truly observe the effect on '1 core per task' for OpenMP workloads,
# you would typically run an OpenMP-enabled application (e.g., using a library like NumPy/SciPy compiled with OpenMP)
# with these environment variables set *before* launching the Python interpreter or the specific task.


Initial CPU affinity for the current process: {0, 1}

Environment variable OMP_PROC_BIND set to: close
Environment variable OMP_PLACES set to: cores

CPU affinity for the current process after setting OMP_PROC_BIND and OMP_PLACES: {0, 1}


### Simulating Environment Variables Set Before Process Launch

To demonstrate the effect of `OMP_PROC_BIND` and `OMP_PLACES` when they are set *before* a program starts, we can use a subprocess. We'll set these variables in our current Python environment, and then launch a new Python script as a subprocess. This subprocess will inherit the environment variables, and we can check its CPU affinity to see the impact.

In [4]:
import os
import subprocess

# Define the content of the temporary script
script_content = """
import os
import sys

# Print the CPU affinity of this subprocess
print(f"Subprocess CPU affinity: {os.sched_getaffinity(0)}")
print(f"Subprocess OMP_PROC_BIND: {os.environ.get('OMP_PROC_BIND')}")
print(f"Subprocess OMP_PLACES: {os.environ.get('OMP_PLACES')}")

# Simulate an OpenMP-dependent library call if Qiskit-Aer is installed
try:
    from qiskit_aer import AerSimulator
    print("Qiskit-Aer imported successfully in subprocess.")
except ImportError:
    print("Qiskit-Aer not available in subprocess.")
"""

# Write the content to a temporary file
temp_script_name = "check_affinity_subprocess.py"
with open(temp_script_name, "w") as f:
    f.write(script_content)

print(f"Number of CPU cores detected in this environment: {os.cpu_count()}\n")

# Set the environment variables in the current process, which will be inherited by the subprocess
os.environ["OMP_PROC_BIND"] = "close"
os.environ["OMP_PLACES"] = "cores"
print("Environment variables set for current process (will be inherited by subprocess):")
print(f"  OMP_PROC_BIND: {os.environ.get('OMP_PROC_BIND')}")
print(f"  OMP_PLACES: {os.environ.get('OMP_PLACES')}")

# Run the temporary script as a subprocess, ensuring it inherits the current environment
print("\nRunning subprocess to check affinity with inherited environment variables...")
process = subprocess.run(
    ["python", temp_script_name],
    env=os.environ.copy(), # Pass a copy of the current environment
    capture_output=True,
    text=True,
    check=False # Do not raise an error if the subprocess exits with a non-zero code
)

# Print the output from the subprocess
print("\n--- Subprocess Output ---")
print(process.stdout)
if process.stderr:
    print("--- Subprocess Error Output ---")
    print(process.stderr)
print("-------------------------")

# Clean up the temporary script file
os.remove(temp_script_name)
print(f"Temporary script '{temp_script_name}' removed.")


Number of CPU cores detected in this environment: 2

Environment variables set for current process (will be inherited by subprocess):
  OMP_PROC_BIND: close
  OMP_PLACES: cores

Running subprocess to check affinity with inherited environment variables...

--- Subprocess Output ---
Subprocess CPU affinity: {0, 1}
Subprocess OMP_PROC_BIND: close
Subprocess OMP_PLACES: cores
Qiskit-Aer imported successfully in subprocess.

-------------------------
Temporary script 'check_affinity_subprocess.py' removed.


In [5]:
import os
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

# Set OpenMP environment variables before using AerSimulator
# This should influence how AerSimulator's underlying C++ (OpenMP) code behaves.
os.environ["OMP_PROC_BIND"] = "close"
os.environ["OMP_PLACES"] = "cores"
print(f"Set OMP_PROC_BIND: {os.environ.get('OMP_PROC_BIND')}")
print(f"Set OMP_PLACES: {os.environ.get('OMP_PLACES')}\n")

# Create a simple quantum circuit
qc = QuantumCircuit(2, 2) # 2 qubits, 2 classical bits
qc.h(0) # Apply Hadamard gate to qubit 0
qc.cx(0, 1) # Apply CNOT gate with qubit 0 as control and qubit 1 as target
qc.measure([0, 1], [0, 1]) # Measure qubits and map to classical bits

# Select the AerSimulator
simulator = AerSimulator()

# Transpile the circuit for the simulator
transpiled_qc = transpile(qc, simulator)

# Run the simulation with many shots to engage OpenMP if available
print("Running simulation with 100000 shots...")
job = simulator.run(transpiled_qc, shots=100000)
result = job.result()
counts = result.get_counts(transpiled_qc)

print("\nSimulation results (counts):", counts)
print("\nNote: While OMP environment variables are set, os.sched_getaffinity() on the main Python process will still show all available cores, as these variables primarily guide OpenMP's internal thread binding. To observe the '1 core per task' effect, you would typically use system monitoring tools (e.g., 'htop' or 'perf') during the execution of a highly parallel OpenMP workload, or profile the OpenMP library directly.")


Set OMP_PROC_BIND: close
Set OMP_PLACES: cores

Running simulation with 100000 shots...

Simulation results (counts): {'11': 49971, '00': 50029}

Note: While OMP environment variables are set, os.sched_getaffinity() on the main Python process will still show all available cores, as these variables primarily guide OpenMP's internal thread binding. To observe the '1 core per task' effect, you would typically use system monitoring tools (e.g., 'htop' or 'perf') during the execution of a highly parallel OpenMP workload, or profile the OpenMP library directly.
